<a href="https://colab.research.google.com/github/sojiroblade7000-prog/Nigerian-News-Topic-Modeling-Najib/blob/main/Nigerian_News_Topic_Model_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# ==============================================================================
# PROJECT: Nigerian News & Tweets Topic Modeling
# AUTHOR: Najib (sojiroblade7000-prog)
# TECH STACK: Python, Pandas, Scikit-Learn
# ==============================================================================

import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

def load_data():
    """Simulates/loads Nigerian news articles and social media tweets."""
    raw_dataset = {
        'text': [
            "CBN increases interest rate again to curb inflation and stabilize the Naira.",
            "Fuel scarcity hits Lagos and Abuja as NNPC explains supply chain bottlenecks.",
            "Super Eagles win AFCON qualifier match against South Africa in dramatic fashion.",
            "Federal Government announces new infrastructure fund for national grid expansion.",
            "Tech startups in Yaba raise millions in seed funding for cross-border payments.",
            "Naira is dropping again o, how are people surviving this high inflation? #Economy",
            "No fuel for light, gen is running on empty. NNPC make una fix this fuel scarcity.",
            "Victor Osimhen scored a masterpiece today! What a win for the Super Eagles!",
            "Light don go again. Band A users paying high electricity tariff for few hours.",
            "Fintech in Lagos is booming, another startup just secured Series A funding!"
        ],
        'source': ['News', 'News', 'News', 'News', 'News', 'Tweet', 'Tweet', 'Tweet', 'Tweet', 'Tweet']
    }
    return pd.DataFrame(raw_dataset)

def clean_text(text):
    """Preprocesses raw text by removing links, handles, punctuation, and noise."""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)    # Remove URLs
    text = re.sub(r'@\w+', '', text)                      # Remove Mentions
    text = re.sub(r'#\w+', '', text)                      # Remove Hashtags
    text = re.sub(r'[^a-zA-Z\s]', '', text)                # Remove Non-Alphabet characters
    text = re.sub(r'\s+', ' ', text).strip()              # Clean extra whitespace
    return text

def get_nigerian_stopwords():
    """Combines standard English stop words with localized Nigerian noise terms."""
    standard_stopwords = [
        'i', 'me', 'my', 'we', 'our', 'you', 'he', 'she', 'it', 'they', 'them',
        'what', 'which', 'who', 'this', 'that', 'am', 'is', 'are', 'was', 'were',
        'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'a',
        'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while',
        'of', 'at', 'by', 'for', 'with', 'about', 'to', 'from', 'in', 'out', 'on', 'again'
    ]
    nigerian_noise = [
        'nigeria', 'nigerian', 'lagos', 'abuja', 'said', 'today', 'people',
        'make', 'una', 'don', 'dey', 'go', 'o', 'just'
    ]
    return standard_stopwords + nigerian_noise

def run_topic_model(df, num_topics=4):
    """Extracts topics using TF-IDF Vectorization and NMF Matrix Factorization."""
    # Clean text
    df['clean_text'] = df['text'].apply(clean_text)

    # Feature extraction
    vectorizer = TfidfVectorizer(
        stop_words=get_nigerian_stopwords(),
        ngram_range=(1, 2)
    )
    tfidf_matrix = vectorizer.fit_transform(df['clean_text'])

    # Fit NMF model
    nmf_model = NMF(n_components=num_topics, random_state=42)
    nmf_model.fit(tfidf_matrix)

    # Extract key terms per topic
    feature_names = vectorizer.get_feature_names_out()
    topics = {}
    for idx, topic_vec in enumerate(nmf_model.components_):
        top_words = [feature_names[i] for i in topic_vec.argsort()[:-6:-1]]
        topics[f"Topic {idx + 1}"] = ", ".join(top_words)

    # Assign primary topic back to data
    topic_distributions = nmf_model.transform(tfidf_matrix)
    df['Assigned_Topic'] = topic_distributions.argmax(axis=1) + 1

    return df, topics

if __name__ == "__main__":
    print("--- Starting Nigerian News & Tweets Topic Modeling Pipeline ---")
    data_df = load_data()
    processed_df, identified_topics = run_topic_model(data_df)

    print("\n[IDENTIFIED TOPICS]")
    for topic, terms in identified_topics.items():
        print(f"{topic}: {terms}")

    print("\n[PROCESSED DATA SAMPLES]")
    print(processed_df[['source', 'text', 'Assigned_Topic']])

--- Starting Nigerian News & Tweets Topic Modeling Pipeline ---

[IDENTIFIED TOPICS]
Topic 1: fuel, nnpc, scarcity, fuel scarcity, supply chain
Topic 2: win, eagles, super, super eagles, scored
Topic 3: naira, inflation, high, surviving high, naira dropping
Topic 4: funding, series funding, secured, fintech, booming

[PROCESSED DATA SAMPLES]
  source                                               text  Assigned_Topic
0   News  CBN increases interest rate again to curb infl...               3
1   News  Fuel scarcity hits Lagos and Abuja as NNPC exp...               1
2   News  Super Eagles win AFCON qualifier match against...               2
3   News  Federal Government announces new infrastructur...               4
4   News  Tech startups in Yaba raise millions in seed f...               4
5  Tweet  Naira is dropping again o, how are people surv...               3
6  Tweet  No fuel for light, gen is running on empty. NN...               1
7  Tweet  Victor Osimhen scored a masterpiece to